# Ingesta de publicaciones a Bronze

Ejecutar desde el repositorio en Databricks, con `config/bronze.json` disponible en el mismo árbol de archivos. La carga reemplaza la tabla de publicaciones en cada ejecución, por lo que repetirla no duplica filas.


In [1]:
import csv
import json
from pathlib import Path

try:
    dbutils.widgets.text('project_root', '')
    job_root = dbutils.widgets.get('project_root')
except (NameError, AttributeError):
    job_root = ''

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

config_path = next(
    (parent / "config" / "bronze.json" for parent in ((Path(job_root),) if job_root else (Path.cwd(), *Path.cwd().parents))
     if (parent / "config" / "bronze.json").is_file()),
    None,
)
if config_path is None:
    raise FileNotFoundError("No se encuentra config/bronze.json desde el directorio del notebook")

config = json.loads(config_path.read_text(encoding="utf-8"))
SOURCE_PATH = config["sources"]["publicaciones"]
TARGET_TABLE = config["tables"]["publicaciones"]
expected_columns = list(config["input_schemas"]["publicaciones"])
assert all(dtype.lower() == "string" for dtype in config["input_schemas"]["publicaciones"].values())
SOURCE_SCHEMA = StructType([StructField(name, StringType(), True) for name in expected_columns])


In [3]:
print("Configuración:", config_path)
print("Origen:", SOURCE_PATH)
print("Tabla:", TARGET_TABLE)


Configuración: c:\Users\Usuario\Downloads\LensData\config\bronze.json
Origen: /Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
Tabla: workspace.lensdata_bronze.publicaciones


In [2]:
# Comparar la línea real antes de aplicar el esquema: Spark puede asociar columnas por posición.
header_line = spark.read.text(SOURCE_PATH).first()[0].lstrip("\ufeff")
actual_columns = next(csv.reader([header_line]))
assert actual_columns == expected_columns, (
    f"Cabecera inesperada en {SOURCE_PATH}: {actual_columns}; esperada: {expected_columns}"
)


Bronze conserva los campos como cadenas, incluidos espacios finales. El tipado y la normalización corresponden a Silver. La cabecera se verifica antes de sobrescribir la tabla.


In [5]:
publicaciones = (
    spark.read
    .option("header", True)
    .option("mode", "FAILFAST")
    .option("ignoreLeadingWhiteSpace", False)
    .option("ignoreTrailingWhiteSpace", False)
    .schema(SOURCE_SCHEMA)
    .csv(SOURCE_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

assert publicaciones.count() == 40, "Se esperaban 40 publicaciones en el CSV"
assert publicaciones.filter(F.col("categoria") == "Regalos ").count() >= 1, "Se perdió el valor original 'Regalos '"
assert publicaciones.filter(
    F.col("_source_file").isNull() |
    (
        F.regexp_replace(F.col("_source_file"), "^dbfs:", "")
        != SOURCE_PATH
    )
).count() == 0, "Procedencia inesperada"
(publicaciones.write.format("delta").mode("overwrite")
 .option("overwriteSchema", True).saveAsTable(TARGET_TABLE))


## Comprobaciones

La tabla debe contener 40 filas, el valor exacto `Regalos ` y la ruta del CSV en `_source_file`. Repetir el notebook debe mantener las 40 filas.


In [7]:
resultado = spark.table(TARGET_TABLE)
assert resultado.count() == 40, "La tabla Bronze no contiene 40 filas"
assert resultado.filter(F.col("categoria") == "Regalos ").count() >= 1, "Falta 'Regalos ' en Bronze"
assert publicaciones.filter(
    F.col("_source_file").isNull() |
    (
        F.regexp_replace(F.col("_source_file"), "^dbfs:", "")
        != SOURCE_PATH
    )
).count() == 0, "Procedencia inesperada"
assert resultado.filter(F.col("_source_file").isNull()).count() == 0, "Falta el archivo de procedencia"
assert all(field.dataType == StringType() for field in resultado.schema if field.name != "_source_file")
display(resultado)


,plataforma,video_id,fecha_publicacion,categoria,titulo,_source_file
0,tiktok,V000,2026-11-01,Regalos,Vídeo de tienda 1,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
1,instagram,V000,2026-11-01,comida asiática,Vídeo de tienda 2,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
2,tiktok,V001,2026-10-31,comida asiática,Vídeo de tienda 3,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
3,instagram,V001,2026-10-31,belleza,Vídeo de tienda 4,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
4,tiktok,V002,2026-10-30,belleza,Vídeo de tienda 5,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
5,instagram,V002,2026-10-30,juguetes,Vídeo de tienda 6,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
6,tiktok,V003,2026-11-01,juguetes,Vídeo de tienda 7,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
7,instagram,V003,2026-11-01,regalos,Vídeo de tienda 8,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
8,tiktok,V004,2026-10-31,regalos,Vídeo de tienda 9,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
9,instagram,V004,2026-10-31,comida asiática,Vídeo de tienda 10,dbfs:/Volumes/workspace/lensdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv
